# CareLoop Clinics — Data preparation

This notebook will hold Part 1 of the project. We start with Task 1.1 only.

## Task 1.1 — Read and type-check the raw file

The CSV file stores every value as text. We use Python's built-in `csv` module to read it and convert the four numeric columns. A row that cannot be converted goes into a rejected list so we can inspect it; this is a type check, not yet a decision that the booking is valid or invalid.

Put the original file at `data/appointments.csv` before running this cell. Keep the original file unchanged.

In [ ]:
import csv
import json

# Read the CSV as dictionaries so each value is linked to its column name.
with open("data/appointments.csv", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    print("Header:", reader.fieldnames)
    raw_rows = list(reader)

print("Rows read:", len(raw_rows))

def parse_row(row):
    """Return a copy of one row with numeric values converted to numbers."""
    converted = dict(row)
    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])
    return converted

accepted = []
rejected = []

for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        # Keep the original text values so we can see what failed later.
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Saved rejected rows to rejected_rows.json")

## What to notice

- `DictReader` uses the header row as dictionary keys, so we can ask for `row["age"]` instead of remembering a column position.
- `int()` and `float()` turn numeric text into numbers. A blank value cannot be converted, so Python raises `ValueError`.
- `try`/`except` lets the loop continue after a bad row instead of stopping the whole notebook.
- The expected result for the supplied file is 1,020 rows read, 988 accepted, and 32 rejected. We will compare our run with those numbers once the data files are added. The rejected rows are for inspection; later pandas cleaning starts again from the original CSV.

## Task 1.2 — Inspect and clean, pass 1

This task starts again from the original CSV; it does not use Task 1.1's `accepted` list. First we inspect the data so we know what needs attention. Then we remove exact duplicate rows, standardize channel labels, convert the date columns, and remove rows that have no usable appointment date or status.

Why these steps matter: duplicates can inflate counts, inconsistent labels split one category into several, and dates/status are needed for later analysis.

In [ ]:
import pandas as pd

# Start fresh from the original file, as Task 1.2 requires.
df = pd.read_csv("data/appointments.csv")

# Inspect the raw data before making changes.
print("Raw shape:", df.shape)
df.info()
print("\nMissing values by column:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("Distinct booking_channel values before cleaning:", df["booking_channel"].nunique())

# (a) Remove rows that are exact duplicates.
df = df.drop_duplicates()
print("\nRows after removing duplicates:", len(df))

# (b) Trim extra spaces and standardize capitalization in the labels.
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Rows after cleaning booking_channel:", len(df))
print("Distinct booking_channel values after cleaning:", df["booking_channel"].nunique())

# (c) Convert text dates to datetime values. Unreadable appointment dates become NaT.
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("Rows after converting dates:", len(df))

# (d) Remove rows missing the appointment date or attendance status.
df = df.dropna(subset=["appt_date", "status"])
print("Rows after dropping missing appt_date or status:", len(df))

### Task 1.2 results to compare

The brief's reference results are: raw shape `(1020, 14)`, 20 exact duplicate rows, 1,000 rows after deduplication, and 990 rows after dropping missing `appt_date` or `status`. The channel labels should go from 16 distinct spellings to 3. Run the cell yourself and compare your output with these reference values.

## Task 1.3 — Clean, pass 2: validity, gaps and outliers

Continue from the cleaned `df` produced in Task 1.2. First mark ages outside 1–100 as missing, then fill missing ages and distances with their medians. Only after filling those gaps, calculate the IQR fences for `distance_km` and `fee_inr`, cap values at the fences, and apply the requested rounding.

In [ ]:
import numpy as np

# (a) Treat ages outside the valid range as missing.
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

# (b) Fill gaps before calculating quartiles and fences.
df["distance_km"] = pd.to_numeric(df["distance_km"], errors="coerce")
df["fee_inr"] = pd.to_numeric(df["fee_inr"], errors="coerce")
age_median = df["age"].median()
distance_median = df["distance_km"].median()
print("Median age:", age_median)
print("Median distance_km:", distance_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

# (c) Compute the IQR fences and cap outliers with np.clip.
for column in ["distance_km", "fee_inr"]:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    capped_count = ((df[column] < lower_fence) | (df[column] > upper_fence)).sum()
    print(f"{column}: Q1={q1}, Q3={q3}, lower fence={lower_fence}, upper fence={upper_fence}")
    df[column] = np.clip(df[column], lower_fence, upper_fence)
    print(f"{column}: capped {capped_count} value(s)")

# (d) Round to the requested precision.
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest fee_inr after capping and rounding:", df["fee_inr"].max())

print("\nFinal shape:", df.shape)
print("Missing values in age, distance_km, fee_inr:")
print(df[["age", "distance_km", "fee_inr"]].isna().sum())

## Task 1.4 — Features and a NumPy summary

Continue with the cleaned `df` from Task 1.3. Create `lead_days` from the difference between appointment and booking dates, and encode `no_show` as 1 for `"No-Show"` and 0 otherwise. Then convert the four requested columns to a NumPy array and calculate the summaries without loops.

In [ ]:
# Create the requested features.
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

# Convert the selected columns to a NumPy array.
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Mean of each column.
print("Mean by column [age, distance_km, fee_inr, lead_days]:")
print(np.mean(X, axis=0))

# (b) Count and share of rows whose lead time exceeds 14 days.
mask = X[:, 3] > 14
print("Rows with lead_days > 14:", mask.sum())
print("Share of rows with lead_days > 14:", mask.mean())

# (c) Distance z-scores and the count beyond two standard deviations.
distance = X[:, 1]
z_scores = (distance - np.mean(distance)) / np.std(distance)
print("Rows with |distance_km z-score| > 2:", (np.abs(z_scores) > 2).sum())

## Task 1.5 — Enrich clinics from Open-Meteo

This task needs clinic coordinates. The appointments file contains clinic IDs but no names or latitude/longitude, so provide `data/clinics.csv` with eight rows and columns `clinic_id`, `clinic_name`, `latitude`, and `longitude`. The code retries each failed request once, records `None` when it cannot get a temperature, and saves the clinic columns plus `temp_c` to `clinics_enriched.csv`.

In [ ]:
from pathlib import Path
import time

import requests

clinics_file = Path("data/clinics.csv")
if not clinics_file.exists():
    raise FileNotFoundError("Add data/clinics.csv with clinic_id, clinic_name, latitude, and longitude for all 8 clinics.")

clinics = pd.read_csv(clinics_file)
required_columns = {"clinic_id", "clinic_name", "latitude", "longitude"}
missing_columns = required_columns - set(clinics.columns)
if missing_columns:
    raise ValueError(f"data/clinics.csv is missing columns: {sorted(missing_columns)}")
if len(clinics) != 8:
    raise ValueError(f"Expected 8 clinic rows, found {len(clinics)}")

def get_temperature(latitude, longitude):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }
    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast", params=params, timeout=10
            )
            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
            else:
                hourly = response.json().get("hourly")
                temperatures = hourly.get("temperature_2m") if hourly else None
                if temperatures:
                    return temperatures[0], None
                reason = "response is missing hourly temperature_2m data"
        except requests.exceptions.RequestException as error:
            reason = f"request failed: {error}"
        except (ValueError, AttributeError) as error:
            reason = f"invalid response JSON: {error}"
        if attempt == 0:
            time.sleep(1)
    return None, reason

temp_values = []
for _, clinic in clinics.iterrows():
    temp_c, reason = get_temperature(clinic["latitude"], clinic["longitude"])
    temp_values.append(temp_c)
    if reason:
        print(f"{clinic['clinic_name']}: {reason}; temp_c set to None")

clinics["temp_c"] = temp_values
clinics.to_csv("clinics_enriched.csv", index=False)
print("Saved clinics_enriched.csv")

## Task 1.6 — Export the clean table and write the SQL load script

This cell uses the cleaned appointment `df` from Tasks 1.2–1.4 and the `clinics` DataFrame from Task 1.5. It validates the 16 appointment columns, sorts by `appointment_id`, formats dates as text, writes `appointments_clean.csv`, then writes clinic INSERTs before appointment INSERTs in `load_data.sql`. Run Task 1.5 successfully first so `clinics_enriched.csv` exists.

In [ ]:
from pathlib import Path

appointment_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show",
]
appointment_numeric = {"age", "distance_km", "fee_inr", "prior_no_shows", "lead_days", "no_show"}
clinic_numeric = {"latitude", "longitude", "temp_c"}

# Sort, format dates as YYYY-MM-DD text, and keep exactly the requested columns.
appointments_export = df.sort_values("appointment_id").copy()
appointments_export["booked_on"] = appointments_export["booked_on"].dt.strftime("%Y-%m-%d")
appointments_export["appt_date"] = appointments_export["appt_date"].dt.strftime("%Y-%m-%d")
appointments_export = appointments_export[appointment_columns]

id_is_unique = appointments_export["appointment_id"].is_unique
columns_have_no_missing = not appointments_export.isna().any().any()
print("appointment_id is unique:", id_is_unique)
print("No missing values in the 16 export columns:", columns_have_no_missing)
if not id_is_unique or not columns_have_no_missing:
    raise ValueError("Appointment export validation failed; files were not written.")

appointments_export.to_csv("appointments_clean.csv", index=False)
clinics_export = pd.read_csv("clinics_enriched.csv")
clinic_columns = list(clinics_export.columns)

def sql_literal(value, column, numeric_columns):
    if pd.isna(value):
        return "NULL"
    if column in numeric_columns:
        if isinstance(value, (int, np.integer)):
            return str(int(value))
        if isinstance(value, (float, np.floating)):
            return str(int(value)) if float(value).is_integer() else format(float(value), ".15g")
        return str(value)
    return "'" + str(value).replace("'", "''") + "'"

sql_statements = []
# Insert clinics first to satisfy the appointments table foreign key.
for _, clinic in clinics_export.iterrows():
    values = ",".join(
        sql_literal(clinic[column], column, clinic_numeric) for column in clinic_columns
    )
    sql_statements.append(f"INSERT INTO clinics VALUES ({values});")

for _, appointment in appointments_export.iterrows():
    values = ",".join(
        sql_literal(appointment[column], column, appointment_numeric)
        for column in appointment_columns
    )
    sql_statements.append(f"INSERT INTO appointments VALUES ({values});")

Path("load_data.sql").write_text("\n".join(sql_statements) + "\n", encoding="utf-8")
print("Saved appointments_clean.csv and load_data.sql")